# Vanishing and Exploding Gradients

> Based on Stanford CS230 — C2M1 + Lecture Notes 3, Andrew Ng / DeepLearning.AI

In very deep networks, gradients can shrink exponentially (vanish) or grow exponentially (explode) as they propagate through layers. This is the central challenge of training deep networks.

## Learning Objectives

1. Explain the mathematical cause of vanishing and exploding gradients
2. Show how weight initialisation (He/Xavier) mitigates the problem at initialisation time
3. Apply gradient clipping to handle exploding gradients at training time
4. Understand why ReLU and proper initialisation together largely solve vanishing gradients

## Mathematical Cause

For a network with identical weight matrices $W$ and linear activations, the gradient at layer 1 is:

$$\frac{\partial J}{\partial A^{[1]}} = (W^T)^L \cdot \frac{\partial J}{\partial A^{[L]}}$$

- If all eigenvalues of $W$ have magnitude $> 1$ → gradients **explode** exponentially
- If all eigenvalues have magnitude $< 1$ → gradients **vanish** exponentially

The deeper the network, the worse the effect.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#f5f5f7', 'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#c8ccd4', 'axes.labelcolor': '#1a1d27',
    'axes.titlecolor': '#1a1d27', 'xtick.color': '#2a2e3a',
    'ytick.color': '#2a2e3a', 'grid.color': '#e0e3ea',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
    'text.color': '#1a1d27', 'font.family': 'DejaVu Sans',
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'legend.facecolor': '#ffffff', 'legend.edgecolor': '#c8ccd4',
    'figure.dpi': 110,
})
P = ['#5b9bd5', '#e05c5c', '#f4b942', '#7ecba1', '#56b6c2', '#c678dd']

# ---- 1. Gradient magnitude across depths (linear network approximation) ----
depths = np.arange(1, 51)
configs = [
    ('Exploding  (w=1.5)', 1.5,   P[1]),
    ('Stable     (w=1.0)', 1.0,   P[3]),
    ('Vanishing  (w=0.7)', 0.7,   P[0]),
    ('Vanishing  (w=0.5)', 0.5,   P[2]),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Vanishing & Exploding Gradients — Depth Effect', fontsize=13, fontweight='bold')

for label, w, color in configs:
    grads = [abs(w) ** d for d in depths]
    axes[0].semilogy(depths, grads, color=color, lw=2.2, label=label)
axes[0].axhline(1, color='#aaa', lw=1, ls='--', label='stable = 1')
axes[0].set_xlabel('Depth (# layers)')
axes[0].set_ylabel('Gradient magnitude (log scale)')
axes[0].set_title('Gradient Growth / Decay with Depth')
axes[0].legend(fontsize=9)
axes[0].grid(True)

# ---- 2. Activation std across layers for different initialisations ----
def activation_stds(n_units, n_layers, scale, activation, seed=0):
    np.random.seed(seed)
    stds = []
    A = np.random.randn(n_units, 1000)
    for _ in range(n_layers):
        W = np.random.randn(n_units, n_units) * scale
        Z = W @ A
        A = activation(Z)
        stds.append(np.std(A))
    return stds

def relu(z): return np.maximum(0, z)
def tanh(z): return np.tanh(z)

n, L = 100, 30
layer_idx = list(range(1, L+1))

init_configs = [
    ('Too small (σ=0.01)',    0.01,              relu,  P[2]),
    ('He  (σ=√2/n, ReLU)',    np.sqrt(2/n),      relu,  P[3]),
    ('Xavier  (σ=1/√n, tanh)', np.sqrt(1/n),     tanh,  P[0]),
    ('Too large (σ=2.0)',      2.0,               relu,  P[1]),
]

for label, scale, act, color in init_configs:
    stds = activation_stds(n, L, scale, act)
    axes[1].semilogy(layer_idx, stds, color=color, lw=2.2, label=label)

axes[1].axhline(1, color='#aaa', lw=1, ls='--', label='std = 1 (ideal)')
axes[1].set_xlabel('Layer index')
axes[1].set_ylabel('Activation std (log scale)')
axes[1].set_title('Activation Std by Layer — Initialisation Matters')
axes[1].legend(fontsize=8)
axes[1].grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# ---- Gradient clipping ----
np.random.seed(7)
n_steps = 200
# Simulate gradient norms during training (some spikes = exploding)
grad_norms_raw = np.abs(np.random.randn(n_steps)) * 2
grad_norms_raw[40:45]  += 80   # simulate gradient explosion
grad_norms_raw[110:113] += 50

clip_threshold = 5.0
grad_norms_clipped = np.minimum(grad_norms_raw, clip_threshold)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('Gradient Clipping — Taming Exploding Gradients', fontsize=13, fontweight='bold')

axes[0].plot(grad_norms_raw,     color=P[1], lw=1.5, label='Raw gradient norm')
axes[0].axhline(clip_threshold,  color=P[2], lw=2, ls='--', label=f'Clip threshold = {clip_threshold}')
axes[0].fill_between(range(n_steps), clip_threshold, grad_norms_raw,
                     where=grad_norms_raw > clip_threshold, color=P[1], alpha=0.25, label='Clipped region')
axes[0].set_xlabel('Training step')
axes[0].set_ylabel('Gradient norm ‖∂J/∂θ‖')
axes[0].set_title('Before Clipping')
axes[0].legend(fontsize=9)
axes[0].grid(True)

axes[1].plot(grad_norms_clipped, color=P[3], lw=1.5, label='Clipped gradient norm')
axes[1].axhline(clip_threshold,  color=P[2], lw=2, ls='--', label=f'Max = {clip_threshold}')
axes[1].set_xlabel('Training step')
axes[1].set_ylabel('Gradient norm ‖∂J/∂θ‖')
axes[1].set_title('After Clipping')
axes[1].legend(fontsize=9)
axes[1].grid(True)

plt.tight_layout()
plt.show()

print("Gradient clipping formula:")
print("  if ‖g‖ > threshold:  g ← g × (threshold / ‖g‖)")
print("  else:                 g unchanged")


## Solutions Summary

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 640 160" width="640" height="160" style="font-family:DejaVu Sans,Arial,sans-serif;background:#f5f5f7;border-radius:8px">
  <!-- Vanishing column -->
  <rect x="20"  y="30" width="280" height="120" rx="6" fill="#5b9bd5" fill-opacity="0.08" stroke="#5b9bd5" stroke-width="1.5"/>
  <text x="160" y="52" text-anchor="middle" fill="#3a7bbf" font-size="12" font-weight="bold">Vanishing Gradients</text>
  <text x="35"  y="73" fill="#444" font-size="11">✓ Use ReLU (gradient = 1 for z &gt; 0)</text>
  <text x="35"  y="92" fill="#444" font-size="11">✓ He / Xavier initialisation</text>
  <text x="35"  y="111" fill="#444" font-size="11">✓ Batch normalisation (next notebook)</text>
  <text x="35"  y="130" fill="#444" font-size="11">✓ Residual connections (skip connections)</text>
  <!-- Exploding column -->
  <rect x="340" y="30" width="280" height="120" rx="6" fill="#e05c5c" fill-opacity="0.08" stroke="#e05c5c" stroke-width="1.5"/>
  <text x="480" y="52" text-anchor="middle" fill="#b03a3a" font-size="12" font-weight="bold">Exploding Gradients</text>
  <text x="355" y="73" fill="#444" font-size="11">✓ Gradient clipping (clip by norm/value)</text>
  <text x="355" y="92" fill="#444" font-size="11">✓ He / Xavier initialisation</text>
  <text x="355" y="111" fill="#444" font-size="11">✓ Batch normalisation</text>
  <text x="355" y="130" fill="#444" font-size="11">✓ Smaller learning rate</text>
  <!-- centre label -->
  <text x="320" y="18" text-anchor="middle" fill="#666" font-size="10" font-style="italic">— both problems occur in very deep networks —</text>
</svg>
